# Dataset cleaning and leakage audit
This notebook measures and reports data-quality problems without inventing metrics. It writes JSON and CSV reports plus a cleaned JSONL artifact.

In [ ]:
import csv
import difflib
import hashlib
import json
import os
import re
from collections import Counter
from pathlib import Path

INPUT_PATH = Path(os.environ.get('AASP_DATASET_INPUT', 'datasets/processed/security_dataset.jsonl'))
OUTPUT_DIR = Path(os.environ.get('AASP_EVALUATION_DIR', 'evaluations'))
CLEAN_PATH = Path(os.environ.get('AASP_CLEAN_DATASET_OUTPUT', 'datasets/processed/security_dataset.cleaned.jsonl'))
MAX_CHARACTERS = int(os.environ.get('AASP_MAX_SAMPLE_CHARACTERS', '50000'))
NEAR_DUPLICATE_THRESHOLD = float(os.environ.get('AASP_NEAR_DUPLICATE_THRESHOLD', '0.92'))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CLEAN_PATH.parent.mkdir(parents=True, exist_ok=True)
if not INPUT_PATH.exists():
    raise FileNotFoundError(f'Dataset not found: {INPUT_PATH.resolve()}')
print(f'Analyzing {INPUT_PATH.resolve()}')

In [ ]:
VALID_SEVERITIES = {'INFO', 'LOW', 'MEDIUM', 'HIGH', 'CRITICAL'}
CWE_PATTERN = re.compile(r'^CWE-\d+$')
records = []
issues = []
malformed_lines = 0
for line_number, line in enumerate(INPUT_PATH.read_text(encoding='utf-8').splitlines(), start=1):
    if not line.strip():
        issues.append({'sample_id': '', 'issue': 'EMPTY_LINE', 'detail': f'Line {line_number}'})
        continue
    try:
        record = json.loads(line)
    except json.JSONDecodeError as exc:
        malformed_lines += 1
        issues.append({'sample_id': '', 'issue': 'MALFORMED_JSON', 'detail': f'Line {line_number}: {exc}'})
        continue
    records.append(record)

def canonical_payload(record):
    return json.dumps({'task_type': record.get('task_type'), 'evidence': record.get('evidence'), 'expected': record.get('expected')}, sort_keys=True, separators=(',', ':'))

def evidence_text(record):
    return json.dumps(record.get('evidence', {}), sort_keys=True).lower()

seen_hashes = {}
valid_records = []
text_by_id = {}
split_hashes = {'train': set(), 'test': set(), 'validation': set()}
for index, record in enumerate(records):
    sample_id = record.get('sample_id') if isinstance(record, dict) else ''
    sample_id = sample_id if isinstance(sample_id, str) else ''
    record_issues = []
    if not isinstance(record, dict) or not sample_id.strip():
        record_issues.append(('EMPTY_OR_INVALID_SAMPLE', 'Record requires a non-empty sample_id.'))
    evidence = record.get('evidence') if isinstance(record, dict) else None
    expected = record.get('expected') if isinstance(record, dict) else None
    if not isinstance(evidence, dict) or not evidence:
        record_issues.append(('MISSING_EVIDENCE', 'Evidence is missing or empty.'))
    if not isinstance(expected, dict):
        record_issues.append(('MISSING_EXPECTED', 'Expected result is missing.'))
        expected = {}
    severity = expected.get('severity')
    if severity not in VALID_SEVERITIES:
        record_issues.append(('INVALID_SEVERITY', str(severity)))
    cwe_values = expected.get('cwe')
    if not isinstance(cwe_values, list) or not all(isinstance(value, str) and CWE_PATTERN.fullmatch(value) for value in cwe_values):
        record_issues.append(('INVALID_CWE', str(cwe_values)))
    owasp_values = expected.get('owasp')
    if not isinstance(owasp_values, list) or not all(isinstance(value, str) and value.strip() for value in owasp_values):
        record_issues.append(('INVALID_OWASP', str(owasp_values)))
    serialized = json.dumps(record, sort_keys=True)
    if len(serialized) > MAX_CHARACTERS:
        record_issues.append(('EXTREMELY_LONG', f'{len(serialized)} characters'))
    payload = canonical_payload(record)
    digest = hashlib.sha256(payload.encode()).hexdigest()
    if digest in seen_hashes:
        record_issues.append(('EXACT_DUPLICATE', f'Duplicates {seen_hashes[digest]}'))
    else:
        seen_hashes[digest] = sample_id
    evidence_lower = evidence_text(record)
    for label in [expected.get('classification'), severity]:
        if isinstance(label, str) and len(label) > 3 and label.lower() in evidence_lower:
            record_issues.append(('POSSIBLE_LABEL_LEAKAGE', f'Expected label appears verbatim in evidence: {label}'))
    split = str(record.get('split', '')).lower()
    if split in split_hashes:
        split_hashes[split].add(digest)
    for issue, detail in record_issues:
        issues.append({'sample_id': sample_id, 'issue': issue, 'detail': detail})
    if not any(issue in {'EMPTY_OR_INVALID_SAMPLE', 'MISSING_EVIDENCE', 'MISSING_EXPECTED', 'INVALID_SEVERITY', 'INVALID_CWE', 'INVALID_OWASP', 'EXACT_DUPLICATE'} for issue, _ in record_issues):
        valid_records.append(record)
        text_by_id[sample_id] = payload
print(f'Parsed {len(records)} record(s); {len(valid_records)} remain after hard validation.')

In [ ]:
near_duplicate_pairs = []
items = list(text_by_id.items())
if len(items) <= 5000:
    for left_index in range(len(items)):
        for right_index in range(left_index + 1, len(items)):
            left_id, left_text = items[left_index]
            right_id, right_text = items[right_index]
            ratio = difflib.SequenceMatcher(None, left_text, right_text, autojunk=False).ratio()
            if ratio >= NEAR_DUPLICATE_THRESHOLD:
                near_duplicate_pairs.append({'left': left_id, 'right': right_id, 'similarity': round(ratio, 6)})
                issues.append({'sample_id': right_id, 'issue': 'NEAR_DUPLICATE', 'detail': f'{left_id} similarity={ratio:.6f}'})
else:
    issues.append({'sample_id': '', 'issue': 'NEAR_DUPLICATE_CHECK_SKIPPED', 'detail': f'{len(items)} records exceeds the exact pairwise limit of 5000.'})

overlap = {
    'train_test': len(split_hashes['train'] & split_hashes['test']),
    'train_validation': len(split_hashes['train'] & split_hashes['validation']),
    'test_validation': len(split_hashes['test'] & split_hashes['validation']),
}
severity_counts = Counter(record.get('expected', {}).get('severity') for record in valid_records)
quality_report = {
    'input_path': str(INPUT_PATH),
    'records_parsed': len(records),
    'records_valid': len(valid_records),
    'malformed_lines': malformed_lines,
    'issue_counts': dict(Counter(item['issue'] for item in issues)),
    'severity_distribution': dict(severity_counts),
    'near_duplicate_threshold': NEAR_DUPLICATE_THRESHOLD,
    'near_duplicate_pairs': near_duplicate_pairs,
    'train_test_overlap': overlap,
    'issues': issues,
}
json_path = OUTPUT_DIR / 'dataset_quality_report.json'
csv_path = OUTPUT_DIR / 'dataset_quality_report.csv'
json_path.write_text(json.dumps(quality_report, indent=2), encoding='utf-8')
with csv_path.open('w', newline='', encoding='utf-8') as handle:
    writer = csv.DictWriter(handle, fieldnames=['sample_id', 'issue', 'detail'])
    writer.writeheader()
    writer.writerows(issues)
with CLEAN_PATH.open('w', encoding='utf-8') as handle:
    for record in valid_records:
        handle.write(json.dumps(record, sort_keys=True) + '\n')
for path in [json_path, csv_path, CLEAN_PATH]:
    if not path.exists():
        raise RuntimeError(f'Artifact validation failed: {path}')
print(json.dumps({key: quality_report[key] for key in ['records_parsed', 'records_valid', 'malformed_lines', 'issue_counts', 'severity_distribution', 'train_test_overlap']}, indent=2))
print(f'Validated artifacts: {json_path.resolve()}, {csv_path.resolve()}, {CLEAN_PATH.resolve()}')